In [ ]:
#Talia and Elijah 
# Load the supermarket dataset
#adding libraries for future use:
# Data manipulation
import pandas as pd
import numpy as np

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Statistical analysis
from scipy import stats

# Multiple linear regression
import statsmodels.api as sm
import statsmodels.formula.api as smf



In [ ]:
#Elijah
filepath = '../data/SuperMarket Analysis.csv'
df = pd.read_csv(filepath)

# Converting transaction dates into datetime format
df['Date'] = pd.to_datetime(df['Date'])

df_time_temp = pd.to_datetime(
    df['Time'],
    format='%I:%M:%S %p',
    errors='coerce'
)


In [ ]:
# talia feature engineering

#making a numerical membership variable
# 1 = Member, 0 = Normal customer
df['Member'] = (df['Customer type'] == 'Member').astype(int)

# making the month variable using elijah's 'date' 
df['Month'] = df['Date'].dt.month

# day of the week variable
df['Day_of_Week'] = df['Date'].dt.day_name()

# adding weekend variable
# 1 = Weekend, 0 = Weekday
df['Is_Weekend'] = df['Date'].dt.dayofweek.isin([5, 6]).astype(int)

# extract the hour of each transaction
df['Hour'] = df_time_temp.dt.hour

print("Dataset Shape:", df.shape)

display(df[[
    'Customer type', 'Member', 'Date',
    'Month', 'Day_of_Week', 'Is_Weekend', 'Hour'
]].head(10))



In [ ]:
# talia checking for missing values
print("Missing Values:")
print(df.isnull().sum())

In [ ]:
#Elijah EDA 
# Calculate average sales for each customer type and product category
category_comparison = df.groupby(
    ['Product line', 'Customer type']
)['Sales'].mean().unstack()

# Calculate the spending difference
# Negative difference means members spend less
category_comparison['Difference'] = (
    category_comparison['Member'] - category_comparison['Normal']
)

# Calculate the percentage difference
category_comparison['Percent_Difference'] = (
    category_comparison['Difference'] /
    category_comparison['Normal']
) * 100

# Sort categories from lowest to highest difference
category_comparison = category_comparison.sort_values('Difference')

print("Member vs Nonmember Spending by Product Category:")
display(category_comparison.round(2))

# Show only categories where members spend less
lower_spending = category_comparison[
    category_comparison['Difference'] < 0
]

print("Categories Where Members Spend Less:")
display(lower_spending.round(2))


In [ ]:
#talia and elijah 
#using data for visualizations 

# Create a bar chart comparing member and nonmember spending
plt.figure(figsize=(12, 6))

sns.barplot(
    data=df,
    x='Sales',
    y='Product line',
    hue='Customer type',
    errorbar=None,
    palette='pastel'
)

plt.title('Average Spending by Product Category and Customer Type')
plt.xlabel('Average Transaction Sales ($)')
plt.ylabel('Product Category')
plt.tight_layout()
plt.show()


In [ ]:
#talia-- t-test 

#need somewhere to store our results
category_results = []

# Testing each product category separately
for category in df['Product line'].unique():

    # getting them from category
    category_data = df[df['Product line'] == category]

    # Separating member and nonmember sales
    member = category_data[
        category_data['Customer type'] == 'Member'
    ]['Sales'].dropna()

    normal = category_data[
        category_data['Customer type'] == 'Normal'
    ]['Sales'].dropna()

    # Only running the test if both groups have enough observations
    if len(member) >= 2 and len(normal) >= 2:

        # Performming Welch's two-sample t-test
        t_stat, p_value = stats.ttest_ind(
            member,
            normal,
            equal_var=False
        )

        # calc average spending difference
        difference = member.mean() - normal.mean()

        # Save the results
        category_results.append([
            category,
            len(member),
            len(normal),
            member.mean(),
            normal.mean(),
            difference,
            p_value
        ])

# Create a table with our results
category_tests = pd.DataFrame(
    category_results,
    columns=[
        'Product Line',
        'Member N',
        'Normal N',
        'Member Average',
        'Normal Average',
        'Difference',
        'P-Value'
    ]
)



In [ ]:
# talia 
#Adjust p-values for testing multiple categories
from statsmodels.stats.multitest import multipletests

category_tests['Adjusted P-Value'] = multipletests(
    category_tests['P-Value'],
    method='holm'
)[1]

# Identify statistically significant differences
category_tests['Significant'] = (
    category_tests['Adjusted P-Value'] < 0.05
)

# Sort by spending difference
category_tests = category_tests.sort_values('Difference')

print("Statistical Significance by Product Category:")
display(category_tests.round(4))


In [ ]:
# talia
#Adjust p-values for testing multiple categories
from statsmodels.stats.multitest import multipletests

category_tests['Adjusted P-Value'] = multipletests(
    category_tests['P-Value'],
    method='holm'
)[1]

# Identify statistically significant differences
category_tests['Significant'] = (
    category_tests['Adjusted P-Value'] < 0.05
)

# Sort by spending difference
category_tests = category_tests.sort_values('Difference')

print("Statistical Significance by Product Category:")
display(category_tests.round(4))

In [ ]:
#talia 
#OVERALL SPENDING DIFFERENCE AND CONFIDENCE INTERVAL

# Separate sales by membership
member_sales = df[df['Member'] == 1]['Sales'].dropna()
normal_sales = df[df['Member'] == 0]['Sales'].dropna()

# Calculate average sales
member_mean = member_sales.mean()
normal_mean = normal_sales.mean()

# Calculate the difference
difference = member_mean - normal_mean

# Perform Welch's t-test
result = stats.ttest_ind(
    member_sales,
    normal_sales,
    equal_var=False
)

# Calculate the 95% confidence interval
confidence_interval = result.confidence_interval(confidence_level=0.95)

print("Member Average:", round(member_mean, 2))
print("Nonmember Average:", round(normal_mean, 2))
print("Average Difference:", round(difference, 2))
print("P-Value:", round(result.pvalue, 4))
print("95% Confidence Interval:", confidence_interval)

# Interpret the results
if result.pvalue < 0.05:
    print("There is a statistically significant difference in average spending.")
else:
    print("There is not enough evidence of a statistically significant difference.")


In [ ]:
#talia & Elijah 
#LOYALTY PROGRAM COST-BENEFIT ANALYSIS

# assumptions

# cost to recruit one new loyalty member
cost_per_member = 10

# expected number of additional transactions per new member
extra_transactions = 2

# Estimated profit margin (25%)
profit_margin = 0.25

# expected number of new members
new_members = 100

# Use the current average member transaction value
average_member_sales = member_sales.mean()

# Calculate the cost of recruiting new members
total_cost = cost_per_member * new_members

# Calculate additional revenue
# This assumes the extra transactions are genuinely new purchases
additional_revenue = (
    new_members * extra_transactions * average_member_sales
)

# Calculate additional profit before recruitment costs
additional_profit = additional_revenue * profit_margin

# Calculate net benefit
net_benefit = additional_profit - total_cost

# Calculate return on investment
roi = (net_benefit / total_cost) * 100

# Calculate break-even number of extra transactions per member
break_even = cost_per_member / (
    average_member_sales * profit_margin
)

# Display results
print("Total Program Cost: $", round(total_cost, 2))
print("Additional Revenue: $", round(additional_revenue, 2))
print("Additional Profit Before Recruitment: $", round(additional_profit, 2))
print("Net Benefit: $", round(net_benefit, 2))
print("ROI:", round(roi, 2), "%")
print("Break-Even Extra Transactions per Member:", round(break_even, 2))

# Management recommendation based on the scenario
if net_benefit > 0:
    print("The loyalty program expansion is profitable under these assumptions.")
else:
    print("The loyalty program expansion is not profitable under these assumptions.")


In [ ]:
#Elijah and Talia 
#COST-BENEFIT SCENARIO COMPARISON

# Different costs of recruiting one member
cost_options = [5, 10, 15, 20]

# Different numbers of additional transactions
transaction_options = [1, 2, 3, 4]

# Store results
scenario_results = []

# Calculate the ROI for each combination
for cost in cost_options:
    for transactions in transaction_options:

        # Total recruitment cost
        cost_total = cost * new_members

        # Estimated additional revenue
        revenue = (
            new_members * transactions * average_member_sales
        )

        # Estimated additional profit
        profit = revenue * profit_margin

        # Net benefit
        net = profit - cost_total

        # Return on investment
        scenario_roi = (net / cost_total) * 100

        # Save results
        scenario_results.append([
            cost,
            transactions,
            net,
            scenario_roi
        ])

# Create results table
scenario_table = pd.DataFrame(
    scenario_results,
    columns=[
        'Cost Per Member',
        'Extra Transactions',
        'Net Benefit',
        'ROI (%)'
    ]
)

print("Cost-Benefit Scenarios:")
display(scenario_table.round(2))


In [ ]:
#Elijah 
#COST-BENEFIT HEATMAP-- visualization

# Organize ROI values for the heatmap
roi_table = scenario_table.pivot(
    index='Cost Per Member',
    columns='Extra Transactions',
    values='ROI (%)'
)

# Create the heatmap
plt.figure(figsize=(10, 6))

sns.heatmap(
    roi_table,
    annot=True,
    fmt='.1f',
    cmap='RdYlGn',
    center=0
)

plt.title('Estimated Loyalty Program ROI by Scenario')
plt.xlabel('Additional Transactions per Member')
plt.ylabel('Recruitment Cost per Member ($)')

plt.tight_layout()
plt.show()


In [ ]:
# talia and Elijah 
# advanced linear regress

# Make a copy of the dataset
df_clean = df.copy()

# Replace spaces in column names with underscores
df_clean.columns = [
    col.replace(' ', '_').replace('%', 'pct')
    for col in df_clean.columns
]

# Fit the advanced regression
advanced_model = smf.ols(
    formula=(
        'Sales ~ Member + Quantity + C(Branch) + '
        'C(Product_line) + C(Gender) + '
        'Is_Weekend + Hour'
    ),
    data=df_clean
).fit(cov_type='HC3')

# Display results
print(advanced_model.summary())

# Focus on the membership variable
print("Membership Coefficient:")
print(round(advanced_model.params['Member'], 2))

print("Membership P-Value:")
print(round(advanced_model.pvalues['Member'], 4))


In [ ]:
#Elijah and Talia 
FINAL BUSINESS RECOMMENDATION

print("FINAL SUPERMARKET LOYALTY ANALYSIS")

# Question 1: Where do members spend less?
print("1. WHERE DO MEMBERS SPEND LESS?")

for category in lower_spending.index:
    print(category)

# Question 2: Is the difference significant?
print("2. IS THE OVERALL DIFFERENCE SIGNIFICANT?")

if result.pvalue < 0.05:
    print("Yes. Average spending differs significantly.")
else:
    print("No statistically significant difference was detected.")

# Question 3: Should managers invest more?
print("3. SHOULD MANAGERS INVEST MORE?")

if result.pvalue < 0.05 and difference > 0 and net_benefit > 0:
    print("Membership is associated with higher spending.")
    print("The investment scenario is profitable.")
    print("Recommendation: Test a targeted expansion before scaling.")

elif net_benefit > 0:
    print("The investment scenario is profitable under its assumptions.")
    print("Recommendation: Run a pilot program before expanding.")

else:
    print("The investment scenario does not show a positive return.")
    print("Recommendation: Improve program economics before expanding.")
